# RSNA knee MRI — weighted ResNet34 + DINOv2 submission

Default: **80% ResNet34 + 20% DINOv2**, applied to probabilities for each study/finding. These are experimental weights, not validation-optimized weights; improvement is not guaranteed.

Attach these four inputs on Kaggle:
1. **RSNA Knee Abnormality Detection** competition data.
2. **gany24558/gc-rsna-knee-resnet34 → PyTorch → study-mil** — choose the model version used by your successful ResNet submission.
3. **gany24558/gc-rsna-knee-dinov2 → PyTorch → five-fold-attention** — choose the model version used by your successful DINOv2 submission.
4. **gany24558/rsna-dicom-decoders** — the Python 3.12 Linux GDCM wheel dataset.

Enable a **GPU**, disable **Internet**, start a **fresh session**, and run all cells. No training datasets, reports, API tokens, or original pretrained models are needed. All code is embedded; upload only this notebook.

Outputs: `/kaggle/working/submission.csv` plus component predictions/audits and `hybrid_run_manifest.json`. Submit the top-level `submission.csv` from the completed saved notebook version. Nothing is submitted automatically.


In [ ]:
from pathlib import Path
import time
SESSION_STARTED = time.monotonic()
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working')
RESNET_PACKAGE = None # Explicit manifest.json parent if multiple versions are attached
DINOV2_PACKAGE = None
COMPETITION_ROOT = None
WHEELHOUSE = None
WEIGHTS = {'resnet34': 0.80, 'dinov2': 0.20} # Must sum to 1; same weights for all 12 findings
BATCHES = {'resnet34': 8, 'dinov2': 8}
REQUIRE_GPU = True
MAX_HOURS = 8.5 # Combined budget, including elapsed notebook setup; not per model
FALLBACK_SCORES = [0.5] * 12


## Install the offline DICOM decoder
This runs before either inference runtime imports pydicom. It installs only the attached GDCM wheel, with no network access. If you already imported pydicom, restart the session and Run All.

In [ ]:
# Run before the inference runtime imports pydicom. Requires a fresh session.
import os
import sys
import subprocess
import importlib.util
from pathlib import Path

search_root = Path(WHEELHOUSE) if WHEELHOUSE is not None else INPUT_ROOT
wheels = []
for directory, subdirs, files in os.walk(search_root):
    subdirs[:] = [d for d in subdirs if d not in {'train_series', 'test_series', 'cache', 'features'}]
    wheels.extend(Path(directory)/name for name in files
                  if name.startswith('python_gdcm-3.2.6-') and name.endswith('.whl'))

if wheels:
    if len(wheels) != 1:
        raise ValueError('Multiple GDCM wheels found; set WHEELHOUSE to the intended wheel directory.')
    if 'pydicom.pixels.decoders.gdcm' in sys.modules:
        raise RuntimeError('Restart the Kaggle session, then Run All: pydicom has already cached decoder availability.')
    decoder_dir = Path('/kaggle/working/_dicom_deps')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index',
                           '--no-deps', '--upgrade', '--target', str(decoder_dir), str(wheels[0])])
    sys.path.insert(0, str(decoder_dir))
    import gdcm
    print('Offline GDCM installed:', gdcm.Version.GetVersion())
else:
    print('No attached GDCM wheel found. Existing decoders will be checked before inference.')


## Extract embedded runtimes
The original ResNet34 and DINOv2 runtime sources are retained unchanged. Each model uses its own exported preprocessing, manifest checks and saved-weight tests. They run in separate processes, sequentially, so their Python modules cannot collide and GPU memory is released between models. The installed decoder path is explicitly passed to both processes.

In [ ]:
import sys
CODE_ROOT = OUTPUT_ROOT / '_hybrid_code'
CODE_ROOT.mkdir(parents=True, exist_ok=True)
SOURCES = {'resnet_inference.py': '"""Offline ResNet34 submission using the exact exported validation transforms."""\nimport os\nos.environ[\'HF_HUB_OFFLINE\']=\'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\']=\'1\'\nimport sys, types, json, hashlib, time, csv, traceback, importlib.metadata\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nimport pydicom\nfrom pydicom.errors import InvalidDicomError\nfrom pydicom.pixels import get_decoder\n\nID=\'StudyInstanceUID\';SID=\'SeriesInstanceUID\'\nTARGETS=[\'ACL\',\'MCL\',\'Medial Meniscus\',\'Lateral Meniscus\',\'Medial OA\',\'Lateral OA\',\'PF OA\',\'Effusion\',\'Synovitis\',"Baker\'s",\'Contusion\',\'Fracture\']\nMODEL_HANDLE=\'gany24558/gc-rsna-knee-resnet34/pyTorch/study-mil\'\nSTANDARD_SYNTAXES=[\'1.2.840.10008.1.2\',\'1.2.840.10008.1.2.1\',\'1.2.840.10008.1.2.4.57\',\n                   \'1.2.840.10008.1.2.4.70\',\'1.2.840.10008.1.2.4.90\',\'1.2.840.10008.1.2.4.91\']\n\nclass DecoderUnavailable(Exception):pass\nclass SeriesDataError(Exception):pass\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(8<<20),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef write_json(path,obj):\n    path=Path(path);tmp=path.with_suffix(\'.json.tmp\')\n    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False,default=str));tmp.replace(path)\n\ndef find_files(root,name):\n    for folder,dirs,files in os.walk(root):\n        dirs[:]=[d for d in dirs if d not in {\'train_series\',\'test_series\',\'cache\',\'features\',\'.git\',\'__pycache__\'}]\n        if name in files:yield Path(folder)/name\n\ndef resolve_inputs(input_root,package=None,competition=None):\n    if package is None:\n        matches=[p.parent for p in find_files(input_root,\'manifest.json\')\n                 if \'gc-rsna-knee-resnet34\' in p.parts and \'study-mil\' in p.parts]\n        if len(matches)!=1:raise ValueError(f\'Attach one CNN model version or set MODEL_PACKAGE. Found: {matches}\')\n        package=matches[0]\n    if competition is None:\n        matches=[p.parent for p in find_files(input_root,\'test.csv\') if (p.parent/\'sample_submission.csv\').is_file()]\n        if len(matches)!=1:raise ValueError(f\'Set COMPETITION_ROOT. Found: {matches}\')\n        competition=matches[0]\n    return Path(package),Path(competition)\n\ndef source_module(name,path):\n    module=types.ModuleType(name);module.__file__=str(path)\n    exec(compile(Path(path).read_text(),str(path),\'exec\'),module.__dict__)\n    return module\n\ndef load_package(package,device):\n    package=Path(package).resolve();manifest=json.loads((package/\'manifest.json\').read_text())\n    if manifest.get(\'schema_version\')!=1 or manifest.get(\'model_family\')!=\'resnet34-study-mean-max-v1\':\n        raise ValueError(\'Attach the ResNet34 study-mil package, not the DINO model\')\n    if manifest.get(\'targets\')!=TARGETS:raise ValueError(\'Target order differs from training\')\n    required={\'model.pt\',\'cnn_runtime.py\',\'knee_data.py\',\'native_preprocessing.py\',\'preprocessing_config.json\',\'synthetic_smoke.pt\'}\n    if not required.issubset(manifest[\'files\']):raise ValueError(\'Incomplete exported package\')\n    for name,digest in manifest[\'files\'].items():\n        path=package/name\n        if not path.resolve().is_relative_to(package) or not path.is_file():raise ValueError(\'Invalid package member: \'+name)\n        if sha256(path)!=digest:raise ValueError(\'Package checksum mismatch: \'+name)\n    helpers=source_module(\'resnet_saved_data\',package/\'knee_data.py\')\n    # cnn_runtime imports knee_data. Bind only during import to avoid stale notebook modules.\n    previous=sys.modules.get(\'knee_data\');sys.modules[\'knee_data\']=helpers\n    try:runtime=source_module(\'resnet_saved_runtime\',package/\'cnn_runtime.py\')\n    finally:\n        if previous is None:sys.modules.pop(\'knee_data\',None)\n        else:sys.modules[\'knee_data\']=previous\n    prep=source_module(\'resnet_saved_preprocessing\',package/\'native_preprocessing.py\')\n    cfg=manifest[\'configuration\'];contract=manifest[\'inference\']\n    expected={\'representation\':\'adjacent-slice-triplet\',\'resize\':\'bilinear antialias full-FOV\',\n              \'sampling\':\'rounded linspace\',\'series_selection\':\'all usable, sorted SeriesInstanceUID\',\n              \'slice_pool\':\'mean+max\',\'series_pool\':\'mean\',\'outputs\':\'12 logits; sigmoid once\'}\n    if any(contract.get(k)!=v for k,v in expected.items()):raise ValueError(\'Unsupported CNN input contract\')\n    if cfg[\'image_size\']!=contract[\'image_size\'] or cfg[\'slices_per_series\']!=contract[\'slices_per_series\']:\n        raise ValueError(\'Configuration and inference contract disagree\')\n    if not 1<=int(cfg[\'slices_per_series\'])<=64 or not 32<=int(cfg[\'image_size\'])<=1024:\n        raise ValueError(\'Invalid image/slice dimensions\')\n    prep_identity=json.loads((package/\'preprocessing_config.json\').read_text());prep_cfg=prep_identity[\'config\']\n    if (prep_cfg[\'depth\'],prep_cfg[\'size\'])!=(64,320) or not prep_cfg.get(\'fit_full_fov\'):\n        raise ValueError(\'Unexpected native preprocessing geometry\')\n    saved=torch.load(package/\'model.pt\',map_location=\'cpu\',weights_only=True)\n    if saved[\'architecture\']!=\'resnet34\' or saved[\'dropout\']!=cfg[\'dropout\']:raise ValueError(\'Checkpoint architecture mismatch\')\n    model=runtime.ResNetKnee(pretrained=False,dropout=saved[\'dropout\']).eval()\n    model.load_state_dict(saved[\'state_dict\'],strict=True)\n    np.testing.assert_allclose(model.mean.flatten().numpy(),contract[\'normalization_mean\'],atol=1e-7)\n    np.testing.assert_allclose(model.std.flatten().numpy(),contract[\'normalization_std\'],atol=1e-7)\n    reference=torch.load(package/\'synthetic_smoke.pt\',map_location=\'cpu\',weights_only=True)\n    with torch.inference_mode():actual=model(reference[\'images\']).float().cpu()\n    torch.testing.assert_close(actual,reference[\'expected\'],rtol=1e-4,atol=1e-5)\n    model=model.to(device).eval()\n    # Check the streaming aggregation against the exported forward method on multiple series.\n    images=reference[\'images\']+[reference[\'images\'][0].flip(0)*.9]\n    with torch.inference_mode(),torch.autocast(device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n        direct=model(images).float()\n        vectors=[series_vector(model,x,2,device) for x in images]\n        streamed=model.head(torch.stack(vectors).mean(0,keepdim=True)).float()\n    torch.testing.assert_close(streamed,direct,rtol=2e-3,atol=2e-4)\n    return manifest,runtime,prep,prep_cfg,model\n\ndef decoder_inventory():\n    rows=[]\n    for uid in STANDARD_SYNTAXES:\n        try:\n            decoder=get_decoder(uid)\n            rows.append(dict(uid=uid,available=bool(decoder.is_available),plugins=list(decoder.available_plugins),\n                             missing=list(decoder.missing_dependencies)))\n        except NotImplementedError as error:rows.append(dict(uid=uid,available=False,error=str(error)))\n    return rows\n\ndef safe_uid(value):\n    value=str(value)\n    if not value or value in {\'.\',\'..\'} or \'/\' in value or \'\\\\\' in value or value.strip()!=value:\n        raise ValueError(\'Invalid identifier/path component\')\n    return value\n\ndef sample_schema(frame):\n    if len(frame)==0 or len(frame.columns)!=13 or set(frame.columns)!={ID,*TARGETS}:raise ValueError(\'Invalid sample submission schema\')\n    if frame[ID].isna().any() or frame[ID].duplicated().any():raise ValueError(\'Missing/duplicate sample IDs\')\n    for uid in frame[ID]:safe_uid(uid)\n\ndef load_inputs(competition):\n    sample=pd.read_csv(competition/\'sample_submission.csv\',dtype={ID:str});sample_schema(sample)\n    test=pd.read_csv(competition/\'test.csv\',dtype={ID:str})\n    if test[ID].isna().any() or test[ID].duplicated().any() or set(test[ID])!=set(sample[ID]):\n        raise ValueError(\'test.csv and sample_submission.csv coverage mismatch\')\n    path=competition/\'test_series.csv\';notes={}\n    if path.is_file():\n        frame=pd.read_csv(path,dtype={ID:str,SID:str})\n        if not {ID,SID}.issubset(frame.columns):raise ValueError(\'test_series.csv requires study and series IDs\')\n        notes[\'rows_outside_requested_studies\']=int((~frame[ID].isin(sample[ID])).sum())\n        frame=frame.loc[frame[ID].isin(sample[ID])].copy()\n        if frame[[ID,SID]].isna().any().any():raise ValueError(\'Missing series identifiers\')\n        for uid in frame[SID]:safe_uid(uid)\n        # Exact duplicate rows add no information; conflicting duplicates are audited per series.\n        notes[\'exact_duplicate_rows\']=int(frame.duplicated().sum());frame=frame.drop_duplicates()\n        records=[]\n        for _,g in frame.groupby([ID,SID],sort=True):\n            row=g.iloc[0].to_dict()\n            if len(g)>1:row[\'_metadata_error\']=\'Conflicting duplicate series metadata\'\n            records.append(row)\n    else:\n        notes[\'series_table_missing\']=True;records=[]\n        for uid in sample[ID]:\n            folder=competition/\'test_series\'/uid\n            if folder.is_dir():\n                records.extend({ID:uid,SID:safe_uid(p.name)} for p in sorted(folder.iterdir()) if p.is_dir())\n    groups={uid:[] for uid in sample[ID]}\n    for record in records:groups[record[ID]].append(record)\n    for uid in groups:groups[uid].sort(key=lambda r:r[SID])\n    return sample,groups,notes\n\ndef series_headers(folder,record,infer_plane=True):\n    if record.get(\'_metadata_error\'):raise SeriesDataError(record[\'_metadata_error\'])\n    paths=sorted(folder.glob(\'*.dcm\'))\n    if not paths:raise SeriesDataError(\'No DICOM files in series folder\')\n    first=None;syntaxes=set()\n    for path in paths:\n        header=pydicom.dcmread(path,stop_before_pixels=True)\n        if first is None:first=header\n        uid=str(header.file_meta.TransferSyntaxUID);syntaxes.add(uid)\n    for uid in syntaxes:\n        try:decoder=get_decoder(uid)\n        except NotImplementedError as error:raise SeriesDataError(\'Unsupported transfer syntax: \'+uid) from error\n        if not decoder.is_available:\n            raise DecoderUnavailable(\'Missing decoder for \'+uid+\': \'+\'; \'.join(decoder.missing_dependencies))\n    raw=record.get(\'Anatomical_Plane\');plane=str(raw).strip().capitalize() if pd.notna(raw) else \'\'\n    note=\'\'\n    if plane not in {\'Sagittal\',\'Coronal\',\'Axial\'}:\n        if not infer_plane:raise SeriesDataError(\'Missing/unknown anatomical plane\')\n        iop=np.asarray(first.ImageOrientationPatient,float)\n        if iop.shape!=(6,) or not np.isfinite(iop).all():raise SeriesDataError(\'Cannot infer plane from orientation\')\n        normal=np.cross(iop[:3],iop[3:]);norm=np.linalg.norm(normal)\n        if norm<1e-6:raise SeriesDataError(\'Degenerate slice orientation\')\n        alignment=np.abs(normal/norm);rank=np.argsort(alignment)\n        if alignment[rank[-1]]-alignment[rank[-2]]<.1:raise SeriesDataError(\'Ambiguous anatomical plane\')\n        plane=[\'Sagittal\',\'Coronal\',\'Axial\'][int(rank[-1])]\n        note=\'Plane inferred from DICOM orientation because metadata was missing/unknown\'\n    elif str(raw)!=plane:note=\'Plane spelling/case normalized\'\n    return plane,note,sorted(syntaxes)\n\n@torch.inference_mode()\ndef series_vector(model,images,microbatch,device):\n    features=[];offset=0;batch=int(microbatch)\n    while offset<len(images):\n        # Retry only CUDA allocation failures, reducing this image microbatch.\n        try:\n            with torch.autocast(device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n                x=images[offset:offset+batch].to(device)\n                feature=model.backbone((x-model.mean)/model.std).float()\n            features.append(feature);offset+=len(feature)\n        except torch.cuda.OutOfMemoryError:\n            if device.type!=\'cuda\' or batch<=1:raise\n            if \'x\' in locals():del x\n            torch.cuda.empty_cache();batch=max(1,batch//2)\n    f=torch.cat(features)\n    if not torch.isfinite(f).all():raise ValueError(\'Nonfinite CNN image features\')\n    return torch.cat([f.mean(0),f.max(0).values])\n\n@torch.inference_mode()\ndef predict_study(uid,records,competition,bundle,device,microbatch,fallback,infer_plane,deadline,emit):\n    manifest,runtime,prep,prep_cfg,model=bundle;vectors=[];failed=0\n    for record in records:\n        if time.monotonic()>deadline:raise TimeoutError(\'Inference session time budget reached\')\n        started=time.monotonic();row={ID:uid,SID:record[SID]}\n        folder=competition/\'test_series\'/uid/record[SID]\n        try:\n            plane,note,syntaxes=series_headers(folder,record,infer_plane)\n            image,support,meta=prep.process_series(folder,uid,record[SID],plane,prep_cfg)\n        except DecoderUnavailable:raise\n        except (SeriesDataError,InvalidDicomError,ValueError,OSError,AttributeError,KeyError,RuntimeError) as error:\n            if isinstance(error,RuntimeError) and any(t in str(error).lower() for t in [\'missing dependencies\',\'all plugins are missing\',\'no available plugins\']):\n                raise DecoderUnavailable(str(error)) from error\n            failed+=1;row.update(status=\'skipped\',error_type=type(error).__name__,error=str(error),seconds=time.monotonic()-started)\n            emit(row);continue\n        images=runtime.prepare_series(image,support,meta,manifest[\'configuration\'],rng=None)\n        del image,support\n        vectors.append(series_vector(model,images,microbatch,device));del images\n        row.update(status=\'ok\',seconds=time.monotonic()-started,source_slices=meta[\'input_shape\'][0],\n                   plane=plane,warning=\'; \'.join(x for x in [note,meta.get(\'review_warning\',\'\')] if x),\n                   transfer_syntaxes=\';\'.join(syntaxes))\n        emit(row)\n    if not vectors:return fallback.copy(),dict(status=\'fallback\',usable_series=0,skipped_series=failed)\n    with torch.autocast(device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n        logits=model.head(torch.stack(vectors).mean(0,keepdim=True))\n    pred=logits.float().sigmoid().cpu().numpy()[0]\n    if pred.shape!=(12,) or not np.isfinite(pred).all():raise ValueError(\'Nonfinite CNN prediction\')\n    return pred,dict(status=\'partial\' if failed else \'ok\',usable_series=len(vectors),skipped_series=failed)\n\ndef validate_submission(frame,sample):\n    sample_schema(sample)\n    if list(frame.columns)!=list(sample.columns) or frame[ID].tolist()!=sample[ID].tolist():\n        raise ValueError(\'Submission columns, identifiers or ordering differ from sample\')\n    values=frame[TARGETS].to_numpy(float)\n    if not np.isfinite(values).all() or ((values<0)|(values>1)).any():raise ValueError(\'Invalid submission probabilities\')\n\ndef run_submission(package,competition,output,require_gpu=True,microbatch=8,max_hours=8.5,\n                   fallback_scores=None,infer_missing_plane=True,require_standard_decoders=True):\n    package,competition,output=map(Path,(package,competition,output));output.mkdir(parents=True,exist_ok=True)\n    if (output/\'submission.csv\').exists():\n        (output/\'submission.csv\').replace(output/f\'submission.previous-{time.time_ns()}.csv\')\n    started=time.monotonic();stage=\'preflight\';current=None;studies=[]\n    receipt=dict(status=\'running\',model_handle=MODEL_HANDLE,mounted_package=str(package))\n    write_json(output/\'run_manifest.json\',receipt)\n    try:\n        if microbatch<1 or not 0<max_hours<=8.5:raise ValueError(\'Invalid microbatch/time budget\')\n        fallback=np.array([.5]*12 if fallback_scores is None else fallback_scores,dtype=np.float32)\n        if fallback.shape!=(12,) or not np.isfinite(fallback).all() or ((fallback<0)|(fallback>1)).any():raise ValueError(\'Invalid fallback scores\')\n        device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        if require_gpu and device.type!=\'cuda\':raise RuntimeError(\'Enable a Kaggle GPU accelerator\')\n        torch.manual_seed(42)\n        if device.type==\'cuda\':torch.cuda.reset_peak_memory_stats()\n        sample,groups,notes=load_inputs(competition)\n        decoders=decoder_inventory();write_json(output/\'decoder_report.json\',decoders)\n        missing=[r[\'uid\'] for r in decoders if not r[\'available\']]\n        print(\'Unavailable standard DICOM decoders:\',missing or \'none\',flush=True)\n        if missing:\n            print(\'Attach compatible offline decoder wheels. A missing decoder is not treated as a bad patient scan.\',flush=True)\n            if require_standard_decoders:\n                raise DecoderUnavailable(\'Required scoring-data decoders unavailable: \'+\', \'.join(missing)+\'. See decoder_report.json and the offline wheel setup cell.\')\n        stage=\'model_loading\';bundle=load_package(package,device);manifest=bundle[0]\n        print(f"Loaded best CNN epoch {manifest[\'best_epoch\']}; saved-weight and streaming checks passed.",flush=True)\n        result=sample.copy();result[TARGETS]=np.nan;stage=\'prediction\'\n        fields=[ID,SID,\'status\',\'error_type\',\'error\',\'seconds\',\'source_slices\',\'plane\',\'warning\',\'transfer_syntaxes\']\n        with (output/\'series_audit.csv\').open(\'w\',newline=\'\') as log:\n            writer=csv.DictWriter(log,fieldnames=fields);writer.writeheader()\n            def emit(row):writer.writerow(row);log.flush()\n            for index,uid in enumerate(sample[ID]):\n                current=uid;check=time.monotonic()\n                if check>started+max_hours*3600:raise TimeoutError(\'Inference session time budget reached\')\n                pred,status=predict_study(uid,groups[uid],competition,bundle,device,microbatch,fallback,infer_missing_plane,started+max_hours*3600,emit)\n                result.loc[index,TARGETS]=pred\n                studies.append({ID:uid,**status,\'seconds\':time.monotonic()-check})\n                pd.DataFrame(studies).to_csv(output/\'study_audit.csv\',index=False)\n                elapsed=time.monotonic()-started\n                print(f\'Studies {index+1}/{len(sample)}; {status["status"]}; elapsed {elapsed/60:.1f} min; rough total {elapsed/(index+1)*len(sample)/60:.1f} min\',flush=True)\n        stage=\'export\';validate_submission(result,sample)\n        temp=output/\'submission.csv.tmp\';result.to_csv(temp,index=False)\n        validate_submission(pd.read_csv(temp,dtype={ID:str}),sample);temp.replace(output/\'submission.csv\')\n        receipt.update(status=\'complete\',studies=len(sample),usable_series=sum(r[\'usable_series\'] for r in studies),\n            skipped_series=sum(r[\'skipped_series\'] for r in studies),fallback_studies=sum(r[\'status\']==\'fallback\' for r in studies),\n            partial_studies=sum(r[\'status\']==\'partial\' for r in studies),fallback_scores=fallback.tolist(),\n            input_notes=notes,package_manifest_sha256=sha256(package/\'manifest.json\'),best_epoch=manifest[\'best_epoch\'],\n            submission_sha256=sha256(output/\'submission.csv\'),device=str(device),microbatch=microbatch,\n            peak_gpu_bytes=torch.cuda.max_memory_allocated() if device.type==\'cuda\' else 0,\n            infer_missing_plane=infer_missing_plane,seconds=time.monotonic()-started,\n            environment={k:importlib.metadata.version(k) for k in [\'torch\',\'torchvision\',\'numpy\',\'pandas\',\'pydicom\',\'scipy\']})\n        write_json(output/\'run_manifest.json\',receipt)\n        print(\'Validated submission saved:\',output/\'submission.csv\',flush=True)\n        return result,receipt\n    except Exception as error:\n        receipt.update(status=\'failed\',stage=stage,study=current,completed_studies=len(studies),\n                       error_type=type(error).__name__,error=str(error),seconds=time.monotonic()-started)\n        write_json(output/\'run_manifest.json\',receipt)\n        (output/\'failure_traceback.txt\').write_text(traceback.format_exc())\n        raise\n', 'dinov2_inference.py': '"""Offline inference using the training package\'s exact adapters and transforms."""\nimport os\nos.environ[\'HF_HUB_OFFLINE\'] = \'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\'] = \'1\'\nimport importlib.util, importlib.metadata, hashlib, json, time\nfrom contextlib import nullcontext\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom pydicom.errors import InvalidDicomError\n\nID = \'StudyInstanceUID\'\nSID = \'SeriesInstanceUID\'\nTARGETS = [\'ACL\',\'MCL\',\'Medial Meniscus\',\'Lateral Meniscus\',\'Medial OA\',\'Lateral OA\',\n           \'PF OA\',\'Effusion\',\'Synovitis\',"Baker\'s",\'Contusion\',\'Fracture\']\nMODEL_HANDLE = \'gany24558/gc-rsna-knee-dinov2/pyTorch/five-fold-attention\'\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(8 << 20), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\ndef json_write(path, value):\n    path = Path(path)\n    temp = path.with_suffix(\'.json.tmp\')\n    temp.write_text(json.dumps(value, indent=2, allow_nan=False, default=str))\n    temp.replace(path)\n\ndef discover(root, filename):\n    found = []\n    for folder, dirs, files in os.walk(root):\n        dirs[:] = [d for d in dirs if d not in {\'train_series\',\'test_series\',\'.cache\'}]\n        if filename in files:\n            found.append(Path(folder) / filename)\n    return found\n\ndef unique_path(candidates, description):\n    if len(candidates) != 1:\n        raise ValueError(f\'Set the explicit {description} path. Candidates: {candidates}\')\n    return candidates[0]\n\ndef load_module(name, path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    module = importlib.util.module_from_spec(spec)\n    # Execute verified source without writing __pycache__ into the model package.\n    exec(compile(Path(path).read_text(), str(path), \'exec\'), module.__dict__)\n    return module\n\ndef validate_submission(frame, sample):\n    if list(frame.columns) != [ID, *TARGETS] or list(sample.columns) != [ID, *TARGETS]:\n        raise ValueError(\'Submission columns do not match the twelve-target contract\')\n    if frame[ID].isna().any() or frame[ID].duplicated().any():\n        raise ValueError(\'Missing or duplicate submission IDs\')\n    if frame[ID].tolist() != sample[ID].tolist():\n        raise ValueError(\'Submission IDs/order differ from sample_submission.csv\')\n    values = frame[TARGETS].to_numpy(float)\n    if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():\n        raise ValueError(\'Invalid prediction probabilities\')\n\ndef load_package(package, device):\n    package = Path(package)\n    manifest = json.loads((package/\'manifest.json\').read_text())\n    if manifest.get(\'schema_version\') != 1 or manifest.get(\'runtime_version\') != \'rsna-frozen-dino-v1\':\n        raise ValueError(\'Unsupported model package version\')\n    if manifest[\'targets\'] != TARGETS or not manifest[\'complete_oof\']:\n        raise ValueError(\'Wrong targets or incomplete OOF package\')\n    if sorted(f[\'fold\'] for f in manifest[\'folds\']) != list(range(5)):\n        raise ValueError(\'Expected exactly five distinct folds\')\n    required = {\'knee_runtime.py\',\'native_preprocessing.py\',\'preprocessing_config.json\',\n                \'synthetic_head_smoke.pt\',\'encoder/adapter.json\'}\n    required.update(f[\'file\'] for f in manifest[\'folds\'])\n    if not required.issubset(manifest[\'files\']):\n        raise ValueError(\'Missing required manifest entries\')\n    actual = {str(p.relative_to(package)) for p in package.rglob(\'*\') if p.is_file()}\n    if actual != set(manifest[\'files\']) | {\'manifest.json\'}:\n        raise ValueError(\'Unexpected or missing package files\')\n    for name, digest in manifest[\'files\'].items():\n        path = package/name\n        if path.is_symlink() or not path.resolve().is_relative_to(package.resolve()):\n            raise ValueError(\'Unsafe model package path\')\n        if sha256(path) != digest:\n            raise ValueError(\'Model checksum mismatch: \'+name)\n    runtime = load_module(\'saved_knee_runtime\', package/\'knee_runtime.py\')\n    prep = load_module(\'saved_knee_preprocessing\', package/\'native_preprocessing.py\')\n    prep_identity = json.loads((package/\'preprocessing_config.json\').read_text())\n    config = prep_identity[\'config\']\n    if config[\'depth\'] != 64 or config[\'size\'] != 320 or not config.get(\'fit_full_fov\'):\n        raise ValueError(\'Unexpected preprocessing image contract\')\n    cfg = manifest[\'configuration\']\n    if cfg[\'representation\'] not in {\'single\',\'triplet\'} or not 1 <= cfg[\'centers\'] <= 64:\n        raise ValueError(\'Invalid slice sampling contract\')\n    heads = []\n    for fold in manifest[\'folds\']:\n        saved = torch.load(package/fold[\'file\'], map_location=\'cpu\', weights_only=True)\n        if saved[\'fold\'] != fold[\'fold\'] or sha256(package/fold[\'file\']) != fold[\'sha256\']:\n            raise ValueError(\'Fold identity mismatch\')\n        model = runtime.FindingAttention(saved[\'hidden\']).eval()\n        model.load_state_dict(saved[\'state_dict\'], strict=True)\n        scaler = saved[\'scaler\']\n        mean, std = np.asarray(scaler[\'mean\']), np.asarray(scaler[\'std\'])\n        if scaler[\'transform\'] != \'log_mm\' or mean.shape != (3,) or std.shape != (3,):\n            raise ValueError(\'Unsupported spacing scaler\')\n        if not np.isfinite([mean,std]).all() or (std <= 0).any():\n            raise ValueError(\'Invalid spacing scaler\')\n        heads.append((model, scaler))\n    reference = torch.load(package/\'synthetic_head_smoke.pt\', map_location=\'cpu\', weights_only=True)\n    with torch.inference_mode():\n        actual = heads[0][0](reference[\'x\'],reference[\'protocol\'],reference[\'spacing\'],reference[\'mask\'])\n    torch.testing.assert_close(actual, reference[\'expected\'], rtol=1e-5, atol=1e-6)\n    heads = [(model.to(device), scaler) for model,scaler in heads]\n    encoder = runtime.FrozenEncoder.from_export(package/\'encoder\').to(device).eval()\n    return manifest, runtime, prep, config, encoder, heads\n\ndef predict_study(records, dicom_root, bundle, device, image_batch, fallback=None):\n    manifest, runtime, prep, config, encoder, heads = bundle\n    cfg = manifest[\'configuration\']\n    features, protocols, spacings, audit = [], [], [], []\n    for record in records:\n        started = time.monotonic()\n        try:\n            image, support, meta = prep.process_series(\n                dicom_root/record[ID]/record[SID], record[ID], record[SID],\n                record[\'Anatomical_Plane\'], config)\n        except (InvalidDicomError, ValueError, OSError, RuntimeError, AttributeError) as error:\n            # Only data preparation failures are recoverable; encoder/head errors propagate.\n            message = str(error)\n            if isinstance(error, RuntimeError) and any(term in message.lower() for term in\n                    [\'missing dependencies\', \'no available plugins\', \'all plugins are missing\']):\n                raise RuntimeError(\'A DICOM decoder dependency is missing; attach offline decoder wheels. \' + message) from error\n            audit.append({ID:record[ID], SID:record[SID], \'status\':\'skipped\',\n                          \'error_type\':type(error).__name__, \'error\':message,\n                          \'seconds\':time.monotonic()-started})\n            continue\n        centers = runtime.sample_centers(int(meta[\'selected_count\']), cfg[\'centers\'])\n        chunks = []\n        for offset in range(0,len(centers),image_batch):\n            x, v = runtime.make_slice_inputs(image, support, centers[offset:offset+image_batch], cfg[\'representation\'])\n            context = torch.autocast(\'cuda\',dtype=torch.float16) if device.type==\'cuda\' else nullcontext()\n            with torch.inference_mode(), context:\n                f = encoder(x.to(device), v.to(device)).float().cpu().numpy()\n            # Training caches slice features as float16 before float32 aggregation.\n            chunks.append(f.astype(np.float16).astype(np.float32))\n        f = np.concatenate(chunks)\n        if not np.isfinite(f).all():\n            raise ValueError(\'Nonfinite encoder features\')\n        features.append(np.concatenate([f.mean(0), f.max(0)]))\n        protocols.append(runtime.protocol_codes(record))\n        spacings.append(meta[\'spacing_summary_drc_mm\'])\n        audit.append({ID:record[ID], SID:record[SID], \'status\':\'ok\',\n                      \'source_slices\':meta[\'input_shape\'][0], \'encoded_slices\':len(centers),\n                      \'seconds\':time.monotonic()-started, \'warning\':meta.get(\'review_warning\',\'\')})\n    if not features:\n        if fallback is None:\n            raise ValueError(\'Study has no usable series and no fallback is configured\')\n        audit.append({ID:records[0][ID] if records else \'\', SID:\'\',\n                      \'status\':\'fallback\', \'error\':\'No usable series; fixed fallback scores used\'})\n        return np.asarray(fallback,dtype=np.float32).copy(), audit\n    spacing = np.asarray(spacings, np.float32)\n    if not np.isfinite(spacing).all() or (spacing <= 0).any():\n        raise ValueError(\'Invalid physical spacing\')\n    x = torch.as_tensor(np.asarray(features)[None], device=device)\n    p = torch.as_tensor(np.asarray(protocols)[None], device=device)\n    mask = torch.ones((1,len(features)),dtype=torch.bool,device=device)\n    predictions = []\n    with torch.inference_mode():\n        for model, scaler in heads:\n            scaled = ((np.log(spacing)-np.asarray(scaler[\'mean\']))/np.asarray(scaler[\'std\'])).astype(np.float32)\n            s = torch.as_tensor(scaled[None], device=device)\n            predictions.append(model(x,p,s,mask).sigmoid().float().cpu().numpy()[0])\n    prediction = np.mean(predictions,axis=0)\n    if prediction.shape != (12,) or not np.isfinite(prediction).all():\n        raise ValueError(\'Invalid study prediction\')\n    return prediction, audit\n\ndef run_submission(package, competition, output, require_gpu=True, image_batch=8, max_hours=8.5, fallback=None):\n    if fallback is None:\n        fallback = np.full(12, 0.5, dtype=np.float32)\n    fallback = np.asarray(fallback, dtype=np.float32)\n    if fallback.shape != (12,) or not np.isfinite(fallback).all() or ((fallback<0)|(fallback>1)).any():\n        raise ValueError(\'Fallback must contain twelve finite scores between zero and one\')\n    started = time.monotonic()\n    package, competition, output = map(Path, (package,competition,output))\n    output.mkdir(parents=True, exist_ok=True)\n    # Archive a previous output so a failed new run cannot leave a stale submission.csv.\n    if (output/\'submission.csv\').exists():\n        (output/\'submission.csv\').replace(output/f\'submission.previous-{time.time_ns()}.csv\')\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    if require_gpu and device.type != \'cuda\':\n        raise RuntimeError(\'Enable a Kaggle GPU accelerator before running inference\')\n    torch.manual_seed(42)\n    if device.type == \'cuda\':\n        torch.cuda.reset_peak_memory_stats()\n    test = pd.read_csv(competition/\'test.csv\',dtype={ID:str})\n    sample = pd.read_csv(competition/\'sample_submission.csv\',dtype={ID:str})\n    series = pd.read_csv(competition/\'test_series.csv\',dtype={ID:str,SID:str})\n    validate_submission(sample,sample)\n    if test[ID].isna().any() or test[ID].duplicated().any() or set(test[ID]) != set(sample[ID]):\n        raise ValueError(\'Test/sample study coverage mismatch\')\n    required = {ID,SID,\'Anatomical_Plane\',\'Fluid_Sensitive\',\'Fat_Suppression\'}\n    if not required.issubset(series.columns) or series[[ID,SID]].isna().any().any():\n        raise ValueError(\'Missing series metadata\')\n    if series.duplicated([ID,SID]).any() or not set(series[ID]).issubset(set(test[ID])):\n        raise ValueError(\'Duplicate series or incomplete/unexpected study coverage\')\n    bundle = load_package(package,device)\n    manifest = bundle[0]\n    print(\'Loaded five heads and shared encoder. Exported head smoke test passed.\', flush=True)\n    predictions, audits = [], []\n    groups = {uid: g.sort_values(SID).to_dict(\'records\') for uid,g in series.groupby(ID)}\n    try:\n        for index, uid in enumerate(sample[ID]):\n            if time.monotonic()-started > max_hours*3600:\n                raise TimeoutError(\'Inference exceeded its runtime budget\')\n            pred, audit = predict_study(groups.get(uid, []),competition/\'test_series\',bundle,device,image_batch,fallback)\n            for row in audit:\n                row[ID] = uid\n            predictions.append(pred); audits.extend(audit)\n            pd.DataFrame(audits).to_csv(output/\'processing_audit.csv\',index=False)\n            elapsed = time.monotonic()-started\n            print(f\'Studies {index+1}/{len(sample)}; elapsed {elapsed/60:.1f} min; \'\n                  f\'rough total estimate {elapsed/(index+1)*len(sample)/60:.1f} min\',flush=True)\n    except Exception as error:\n        json_write(output/\'run_manifest.json\',dict(status=\'failed\',model_handle=MODEL_HANDLE,\n                   study=uid,completed_studies=len(predictions),error=str(error),seconds=time.monotonic()-started))\n        raise\n    result = pd.DataFrame(np.stack(predictions),columns=TARGETS)\n    result.insert(0,ID,sample[ID].tolist())\n    validate_submission(result,sample)\n    temp = output/\'submission.csv.tmp\'\n    result.to_csv(temp,index=False)\n    validate_submission(pd.read_csv(temp,dtype={ID:str}),sample)\n    temp.replace(output/\'submission.csv\')\n    receipt = dict(status=\'complete\',model_handle=MODEL_HANDLE,mounted_package=str(package),\n                   package_manifest_sha256=sha256(package/\'manifest.json\'),training_id=manifest[\'training_id\'],\n                   model_folds=[f[\'fold\'] for f in manifest[\'folds\']],studies=len(sample),series=len(series),\n                   seconds=time.monotonic()-started,device=str(device),image_batch=image_batch,\n                   encoder_precision=\'float16 autocast\' if device.type==\'cuda\' else \'float32\',\n                   peak_gpu_bytes=torch.cuda.max_memory_allocated() if device.type==\'cuda\' else 0,\n                   submission_sha256=sha256(output/\'submission.csv\'),\n                   fallback_count=sum(a[\'status\']==\'fallback\' for a in audits),\n                   skipped_series=sum(a[\'status\']==\'skipped\' for a in audits),\n                   fallback_scores=fallback.tolist(),\n                   partial_study_count=len({a[ID] for a in audits if a[\'status\']==\'skipped\'} -\n                                           {a[ID] for a in audits if a[\'status\']==\'fallback\'}),\n                   environment={name:importlib.metadata.version(name) for name in\n                                [\'torch\',\'numpy\',\'pandas\',\'pydicom\',\'scipy\']})\n    json_write(output/\'run_manifest.json\',receipt)\n    print(\'Validated submission saved:\',output/\'submission.csv\')\n    return result, receipt\n', 'hybrid_runtime.py': '"""Sequential offline inference and probability blending; no training or calibration."""\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport numpy as np\nimport pandas as pd\n\nID = \'StudyInstanceUID\'\nTARGETS = [\'ACL\',\'MCL\',\'Medial Meniscus\',\'Lateral Meniscus\',\'Medial OA\',\'Lateral OA\',\n           \'PF OA\',\'Effusion\',\'Synovitis\',"Baker\'s",\'Contusion\',\'Fracture\']\nHANDLES = {\'resnet34\':\'gany24558/gc-rsna-knee-resnet34/pyTorch/study-mil\',\n           \'dinov2\':\'gany24558/gc-rsna-knee-dinov2/pyTorch/five-fold-attention\'}\n\ndef json_write(path, obj):\n    path=Path(path); tmp=path.with_suffix(\'.json.tmp\')\n    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False,default=str));tmp.replace(path)\n\ndef digest(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(8<<20),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef check_weights(weights):\n    if set(weights)!=set(HANDLES):raise ValueError(\'Weights must specify resnet34 and dinov2\')\n    values=[float(weights[k]) for k in HANDLES]\n    if any(not math.isfinite(x) or x<0 for x in values) or not math.isclose(sum(values),1.,abs_tol=1e-8,rel_tol=0):\n        raise ValueError(\'Weights must be finite, nonnegative, and sum to one\')\n    return dict(zip(HANDLES,values))\n\ndef align_predictions(frame,sample):\n    if list(sample.columns)!=[ID,*TARGETS] or sample.empty or sample[ID].isna().any() or sample[ID].duplicated().any():\n        raise ValueError(\'Invalid sample submission schema or study IDs\')\n    if len(frame.columns)!=13 or set(frame.columns)!=set(sample.columns):\n        raise ValueError(\'Prediction label columns differ from sample\')\n    if frame[ID].isna().any() or frame[ID].duplicated().any() or set(frame[ID])!=set(sample[ID]):\n        raise ValueError(\'Prediction studies differ from sample or contain duplicates\')\n    aligned=frame.set_index(ID).loc[sample[ID],TARGETS].to_numpy(dtype=np.float64)\n    if not np.isfinite(aligned).all() or ((aligned<0)|(aligned>1)).any():\n        raise ValueError(\'Predictions must be finite probabilities in [0,1]\')\n    return aligned\n\ndef blend_predictions(frames,sample,weights):\n    weights=check_weights(weights)\n    total=np.zeros((len(sample),12),dtype=np.float64)\n    for name,weight in weights.items():\n        if weight: total+=weight*align_predictions(frames[name],sample)\n    result=pd.DataFrame(total,columns=TARGETS)\n    result.insert(0,ID,sample[ID].tolist())\n    align_predictions(result,sample)\n    return result\n\ndef locate(root,filename):\n    for folder,dirs,files in os.walk(root):\n        dirs[:]=[d for d in dirs if d not in {\'train_series\',\'test_series\',\'cache\',\'features\',\'.git\',\'__pycache__\'}]\n        if filename in files:yield Path(folder)/filename\n\ndef resolve_inputs(root,packages,competition,weights):\n    resolved={}\n    for name,weight in check_weights(weights).items():\n        if not weight:continue\n        value=packages.get(name)\n        if value is None:\n            _,slug,_,variant=HANDLES[name].split(\'/\')\n            candidates=[p.parent for p in locate(root,\'manifest.json\') if slug in p.parts and variant in p.parts]\n            if len(candidates)!=1:raise ValueError(f\'Attach exactly one {name} model version or set its package path. Found {candidates}\')\n            value=candidates[0]\n        value=Path(value)\n        if not (value/\'manifest.json\').is_file():raise FileNotFoundError(f\'Missing {name} manifest: {value}\')\n        resolved[name]=value\n    if competition is None:\n        candidates=[p.parent for p in locate(root,\'test.csv\') if (p.parent/\'test_series.csv\').is_file() and (p.parent/\'sample_submission.csv\').is_file()]\n        if len(candidates)!=1:raise ValueError(f\'Set COMPETITION_ROOT; candidates: {candidates}\')\n        competition=candidates[0]\n    return resolved,Path(competition)\n\nCHILD = \'\'\'import json, sys\nfrom pathlib import Path\ncfg=json.loads(Path(sys.argv[1]).read_text())\nif cfg[\'name\']==\'resnet34\':\n    from resnet_inference import run_submission\n    run_submission(cfg[\'package\'],cfg[\'competition\'],cfg[\'output\'],\n        require_gpu=cfg[\'require_gpu\'],microbatch=cfg[\'batch\'],max_hours=cfg[\'hours\'],\n        fallback_scores=cfg[\'fallback\'],infer_missing_plane=True,require_standard_decoders=True)\nelse:\n    # The same standard decoder preflight also applies to DINO-only runs.\n    from resnet_inference import decoder_inventory, DecoderUnavailable\n    missing=[r[\'uid\'] for r in decoder_inventory() if not r[\'available\']]\n    if missing:raise DecoderUnavailable(\'Missing standard DICOM decoders: \'+\', \'.join(missing))\n    from dinov2_inference import run_submission\n    run_submission(cfg[\'package\'],cfg[\'competition\'],cfg[\'output\'],\n        require_gpu=cfg[\'require_gpu\'],image_batch=cfg[\'batch\'],max_hours=cfg[\'hours\'],fallback=cfg[\'fallback\'])\n\'\'\'\n\ndef run_hybrid(code_root,input_root,output_root,packages=None,competition=None,\n               weights=None,batches=None,require_gpu=True,max_hours=8.5,started=None,\n               fallback=None,decoder_dir=\'/kaggle/working/_dicom_deps\'):\n    started=time.monotonic() if started is None else started\n    output=Path(output_root);output.mkdir(parents=True,exist_ok=True)\n    final=output/\'submission.csv\'\n    if final.exists():final.replace(output/f\'submission.previous-{time.time_ns()}.csv\')\n    receipt={\'status\':\'running\',\'stage\':\'setup\'}\n    json_write(output/\'hybrid_run_manifest.json\',receipt)\n    try:\n        weights=check_weights(weights or {\'resnet34\':.8,\'dinov2\':.2})\n        if not math.isfinite(max_hours) or max_hours<=0:raise ValueError(\'MAX_HOURS must be positive\')\n        batches=batches or {\'resnet34\':8,\'dinov2\':8}\n        if any(type(batches.get(k)) is not int or batches[k]<1 for k,w in weights.items() if w):\n            raise ValueError(\'Image batch sizes must be positive integers\')\n        fallback=[.5]*12 if fallback is None else fallback\n        if np.asarray(fallback).shape!=(12,) or not np.isfinite(fallback).all() or np.any(np.asarray(fallback)<0) or np.any(np.asarray(fallback)>1):\n            raise ValueError(\'Fallback must contain twelve probabilities\')\n        packages,competition=resolve_inputs(input_root,packages or {},competition,weights)\n        sample=pd.read_csv(competition/\'sample_submission.csv\',dtype={ID:str})\n        align_predictions(sample,sample)\n        test=pd.read_csv(competition/\'test.csv\',dtype={ID:str})\n        if test[ID].isna().any() or test[ID].duplicated().any() or set(test[ID])!=set(sample[ID]):\n            raise ValueError(\'Test/sample study coverage differs\')\n        receipt.update(weights=weights,model_handles=HANDLES,packages={k:str(v) for k,v in packages.items()},components={})\n        env=os.environ.copy();env[\'HF_HUB_OFFLINE\']=\'1\';env[\'TRANSFORMERS_OFFLINE\']=\'1\'\n        env[\'PYTHONPATH\']=os.pathsep.join([str(decoder_dir),str(code_root),*sys.path])\n        frames={}\n        for name,weight in weights.items():\n            if not weight:continue\n            remaining=max_hours*3600-(time.monotonic()-started)\n            if remaining<=30:raise TimeoutError(\'Combined runtime budget exhausted before \'+name)\n            receipt[\'stage\']=name;json_write(output/\'hybrid_run_manifest.json\',receipt)\n            component=output/name;component.mkdir(exist_ok=True)\n            config={\'name\':name,\'package\':str(packages[name]),\'competition\':str(competition),\n                    \'output\':str(component),\'require_gpu\':require_gpu,\'batch\':batches[name],\n                    \'hours\':(remaining-15)/3600,\'fallback\':fallback}\n            config_path=component/\'invocation.json\';json_write(config_path,config)\n            print(f\'Running {name}; weight={weight:.2f}; combined budget remaining {remaining/3600:.2f} h\',flush=True)\n            subprocess.run([sys.executable,\'-u\',\'-c\',CHILD,str(config_path)],env=env,check=True,timeout=remaining-15)\n            # Child exit releases the full model/GPU allocation before the other model loads.\n            report=json.loads((component/\'run_manifest.json\').read_text())\n            if report.get(\'status\')!=\'complete\':raise RuntimeError(name+\' did not finish\')\n            frame=pd.read_csv(component/\'submission.csv\',dtype={ID:str})\n            align_predictions(frame,sample);frames[name]=frame;receipt[\'components\'][name]=report\n            json_write(output/\'hybrid_run_manifest.json\',receipt)\n        receipt[\'stage\']=\'blend\'\n        result=blend_predictions(frames,sample,weights)\n        if time.monotonic()-started>=max_hours*3600:raise TimeoutError(\'Combined runtime budget exhausted\')\n        tmp=output/\'submission.csv.tmp\';result.to_csv(tmp,index=False)\n        align_predictions(pd.read_csv(tmp,dtype={ID:str}),sample);tmp.replace(final)\n        receipt.update(status=\'complete\',stage=\'complete\',studies=len(result),seconds=time.monotonic()-started,\n                       submission_sha256=digest(final),fallback_policy=\'Fixed weighted blend, including component fallback scores\')\n        json_write(output/\'hybrid_run_manifest.json\',receipt)\n        print(\'Hybrid submission saved:\',final,flush=True)\n        return result,receipt\n    except Exception as error:\n        # A failed attempt must not leave a current submission artifact.\n        if final.exists():final.replace(output/f\'submission.failed-{time.time_ns()}.csv\')\n        receipt.update(status=\'failed\',error_type=type(error).__name__,error=str(error),seconds=time.monotonic()-started)\n        json_write(output/\'hybrid_run_manifest.json\',receipt)\n        raise\n'}
for name, source in SOURCES.items():
    (CODE_ROOT / name).write_text(source)
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
# Reload the wrapper when rerunning a cell after changing embedded source.
sys.modules.pop('hybrid_runtime', None)
from hybrid_runtime import run_hybrid
print('Embedded inference code ready.')

## Predict, align and blend
Both models process all requested test studies. Predictions are matched by **StudyInstanceUID and finding name**, never blended by unverified row position. Scores remain probabilities; no thresholds, additional sigmoid, prevalence adjustment, or hidden-test calibration are used.

The original per-model policy for invalid series is retained. If a component has no usable series, its declared 0.5 fallback still receives its configured blend weight; inspect its audit counts. Missing decoder/model dependencies remain fatal. If either positive-weight model fails or the combined time budget expires, no current top-level submission is produced. A zero weight skips that model.

Full hidden-test runtime will exceed that of either standalone model and has not been measured. The three example studies do not establish the total scoring time.

In [ ]:
SUBMISSION, RECEIPT = run_hybrid(
    CODE_ROOT, INPUT_ROOT, OUTPUT_ROOT,
    packages={'resnet34': RESNET_PACKAGE, 'dinov2': DINOV2_PACKAGE},
    competition=COMPETITION_ROOT, weights=WEIGHTS, batches=BATCHES,
    require_gpu=REQUIRE_GPU, max_hours=MAX_HOURS, started=SESSION_STARTED,
    fallback=FALLBACK_SCORES,
)
display(SUBMISSION.head())
print({k: RECEIPT[k] for k in ['status', 'studies', 'weights', 'seconds']})
for name, report in RECEIPT['components'].items():
    print(name, 'fallback studies:', report.get('fallback_studies', report.get('fallback_count', 0)),
          'skipped series:', report['skipped_series'])


## Save and submit
Confirm the hybrid receipt says `complete`, inspect both component audit counts, then **Save Version → Save & Run All** with GPU enabled and Internet disabled. Submit `/kaggle/working/submission.csv` (not either component's CSV).

- `resnet34/`: original CNN predictions, decoder report and study/series audits.
- `dinov2/`: original transformer predictions and processing audit.
- `hybrid_run_manifest.json`: weights, mounted model paths, component receipts and output checksum.

Keep outputs private. Choose weights using held-out officially labeled studies unseen during either model's training. Do not assume that a blend improves the successful ResNet baseline.

**Terms:** probability blending is a weighted average of final model scores; this is not a newly trained hybrid architecture. Sequential inference means evaluating one model after the other. A fallback is an explicit emergency score for unusable data, not a calibrated prediction.